In [1]:
import urllib.parse

conn_str = urllib.parse.quote_plus(
    r"DRIVER={ODBC Driver 17 for SQL Server};"
    r"SERVER=SANDIPAN\SQLEXPRESS;"
    r"DATABASE=master;"
    r"Trusted_Connection=yes;"
)

%reload_ext sql
%sql mssql+pyodbc:///?odbc_connect={conn_str}

## 1. SQL Logical Execution Order

**Definition:**

- You write a SQL query starting with `SELECT`. But the database does **not** run it in that order.
- The database always runs a query in this fixed order:
  - `FROM` — pick the table(s)
  - `JOIN` — combine tables, if any
  - `WHERE` — remove rows that don't match
  - `GROUP BY` — put remaining rows into groups
  - `HAVING` — remove whole groups that don't match
  - `SELECT` — pick the columns to show
  - `DISTINCT` — remove duplicate rows, if asked
  - `ORDER BY` — sort the rows
  - `TOP` — cut down the row count (even though you write `TOP` right after `SELECT`, it is actually applied **last**, after sorting)

## 2. SELECT Basics, and TOP

**Definition:**

- `SELECT *` — shows every column.
- `SELECT DISTINCT city` — shows only unique values, no repeats.
- `SELECT COUNT(*)` — counts every row, even ones with NULLs in them.
- `SELECT COUNT(col)` — counts only rows where that column is NOT NULL.
- `TOP N` — limits the result to N rows. Written right after `SELECT`, before the column list.

**Example:**: "Give me the 3 most recent orders"

In [10]:
%%sql
SELECT TOP 3 *
FROM orders

 * mssql+pyodbc:///?odbc_connect=DRIVER%3D%7BODBC+Driver+17+for+SQL+Server%7D%3BSERVER%3DSANDIPAN%5CSQLEXPRESS%3BDATABASE%3Dmaster%3BTrusted_Connection%3Dyes%3B
Done.


row_id,order_id,order_date,ship_date,ship_mode,customer_id,customer_name,segment,country,city,state,postal_code,region,product_id,category,sub_category,product_name,sales,quantity,discount,profit
1.0,CA-2020-152156,2020-11-08,2020-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,42420.0,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.96,2.0,0.0,41.9136
2.0,CA-2020-152156,2020-11-08,2020-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,42420.0,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs, Rounded Back",731.9399999999999,3.0,0.0,219.58199999999997
3.0,CA-2020-138688,2020-06-12,2020-06-16,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,California,90036.0,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters by Universal,14.62,2.0,0.0,6.8713999999999995


## 3. WHERE Clause Operators

**Definition and Example for each operator:**

- `=` — checks if two values are exactly equal.
  ```sql
  WHERE region = 'South'
  ```
- `<>` (or `!=`) — checks if two values are NOT equal.
  ```sql
  WHERE region <> 'South'
  ```
- `>`, `<`, `>=`, `<=` — greater than, less than, and their "or equal to" versions.
  ```sql
  WHERE sales > 1000
  ```
- `BETWEEN ... AND` — checks if a value falls inside a range (both ends included).
  ```sql
  WHERE sales BETWEEN 100 AND 500
  ```
- `IN` — checks if a value matches any one value in a list. Shortcut for several `OR` conditions.
  ```sql
  WHERE region IN ('South', 'West')
  ```
- `NOT IN` — checks if a value does NOT match any value in a list.
  ```sql
  WHERE region NOT IN ('South', 'West')
  ```
- `IS NULL` / `IS NOT NULL` — checks whether a value is missing/empty. In SQL, missing data is called NULL — it is not the same as zero or an empty string.
  ```sql
  WHERE city IS NULL
  ```
- `EXISTS` — checks if a smaller query (a subquery) returns at least one row.
  ```sql
  WHERE EXISTS (SELECT 1 FROM returns WHERE returns.order_id = orders.order_id)
  ```


In [19]:
%%sql
-- Filter South Region customers with discount greater than 0

SELECT TOP 3 *
FROM orders
WHERE region = 'South' AND discount > 0

 * mssql+pyodbc:///?odbc_connect=DRIVER%3D%7BODBC+Driver+17+for+SQL+Server%7D%3BSERVER%3DSANDIPAN%5CSQLEXPRESS%3BDATABASE%3Dmaster%3BTrusted_Connection%3Dyes%3B
Done.


row_id,order_id,order_date,ship_date,ship_mode,customer_id,customer_name,segment,country,city,state,postal_code,region,product_id,category,sub_category,product_name,sales,quantity,discount,profit
4.0,US-2019-108966,2019-10-11,2019-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,Florida,33311.0,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,5.0,0.45,-383.03100000000006
5.0,US-2019-108966,2019-10-11,2019-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,Florida,33311.0,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.368000000000002,2.0,0.2,2.516399999999999
13.0,CA-2021-114412,2021-04-15,2021-04-20,Standard Class,AA-10480,Andrew Allen,Consumer,United States,Concord,North Carolina,28027.0,South,OFF-PA-10002365,Office Supplies,Paper,Xerox 1967,15.552000000000003,3.0,0.2,5.4432


## 4. AND / OR / NOT Precedence

**Definition:**

- `AND` — both conditions must be true.
- `OR` — at least one condition must be true.
- `NOT` — flips true to false and false to true.
- SQL checks `NOT` first, then `AND`, then `OR`. So `A OR B AND C` is actually read as `A OR (B AND C)`, not `(A OR B) AND C`.

**Example:**

- Goal: orders that are (Technology OR Furniture) AND only from the year 2020.

In [18]:
%%sql
SELECT TOP 3 *
FROM orders
WHERE category = 'Technology' OR category = 'Furnitue' AND YEAR(order_date) = 2020

 * mssql+pyodbc:///?odbc_connect=DRIVER%3D%7BODBC+Driver+17+for+SQL+Server%7D%3BSERVER%3DSANDIPAN%5CSQLEXPRESS%3BDATABASE%3Dmaster%3BTrusted_Connection%3Dyes%3B
Done.


row_id,order_id,order_date,ship_date,ship_mode,customer_id,customer_name,segment,country,city,state,postal_code,region,product_id,category,sub_category,product_name,sales,quantity,discount,profit
8.0,CA-2018-115812,2018-06-09,2018-06-14,Standard Class,BH-11710,Brosina Hoffman,Consumer,United States,Los Angeles,California,90032.0,West,TEC-PH-10002275,Technology,Phones,Mitel 5320 IP Phone VoIP phone,907.152,6.0,0.2,90.71520000000004
12.0,CA-2018-115812,2018-06-09,2018-06-14,Standard Class,BH-11710,Brosina Hoffman,Consumer,United States,Los Angeles,California,90032.0,West,TEC-PH-10002033,Technology,Phones,Konftel 250 Conference phone - Charcoal black,911.424,4.0,0.2,68.35680000000002
20.0,CA-2018-143336,2018-08-27,2018-09-01,Second Class,ZD-21925,Zuschuss Donatelli,Consumer,United States,San Francisco,California,94109.0,West,TEC-PH-10001949,Technology,Phones,Cisco SPA 501G IP Phone,213.48000000000002,3.0,0.2,16.01099999999998


- Because `AND` binds tighter, this is really read as:
  `category = 'Technology' OR (category = 'Furniture' AND YEAR(order_date) = 2020)`
- Result: every Technology order leaks in, no matter the year. Only Furniture gets restricted to 2020.

In [20]:
%%sql
SELECT TOP 3 *
FROM orders
WHERE (category = 'Technology' OR category = 'Furnitue') AND YEAR(order_date) = 2020

 * mssql+pyodbc:///?odbc_connect=DRIVER%3D%7BODBC+Driver+17+for+SQL+Server%7D%3BSERVER%3DSANDIPAN%5CSQLEXPRESS%3BDATABASE%3Dmaster%3BTrusted_Connection%3Dyes%3B
Done.


row_id,order_id,order_date,ship_date,ship_mode,customer_id,customer_name,segment,country,city,state,postal_code,region,product_id,category,sub_category,product_name,sales,quantity,discount,profit
27.0,CA-2020-121755,2020-01-16,2020-01-20,Second Class,EH-13945,Eric Hoffmann,Consumer,United States,Los Angeles,California,90049.0,West,TEC-AC-10003027,Technology,Accessories,Imation 8GB Mini TravelDrive USB 2.0 Flash Drive,90.57000000000001,3.0,0.0,11.774100000000004
36.0,CA-2020-117590,2020-12-08,2020-12-10,First Class,GH-14485,Gene Hale,Corporate,United States,Richardson,Texas,75080.0,Central,TEC-PH-10004977,Technology,Phones,GE 30524EE4,1097.5440000000003,7.0,0.2,123.4736999999999
45.0,CA-2020-118255,2020-03-11,2020-03-13,First Class,ON-18715,Odella Nelson,Corporate,United States,Eagan,Minnesota,55122.0,Central,TEC-AC-10000171,Technology,Accessories,"Verbatim 25 GB 6x Blu-ray Single Layer Recordable Disc, 25/Pack",45.98,2.0,0.0,19.7714


## 5. LIKE and Wildcards

**Definition:**

- A wildcard is a symbol that stands in for "any character" or "any text," so you can match patterns instead of exact text.
- MSSQL's `LIKE` supports four wildcard types:
  - `%` — any number of characters (including zero)
  - `_` — exactly one character
  - `[set]` — any one character from the set, e.g. `[SM]` means S or M
  - `[^set]` — any one character NOT in the set
  - `[a-m]` — any one character in a range

**Example:**

- "Names starting with S or M" needs a character-set wildcard, not just `%` or `_`.

In [24]:
%%sql
SELECT TOP 3 *
FROM orders
WHERE customer_name LIKE '[SM]%' 

 * mssql+pyodbc:///?odbc_connect=DRIVER%3D%7BODBC+Driver+17+for+SQL+Server%7D%3BSERVER%3DSANDIPAN%5CSQLEXPRESS%3BDATABASE%3Dmaster%3BTrusted_Connection%3Dyes%3B
Done.


row_id,order_id,order_date,ship_date,ship_mode,customer_id,customer_name,segment,country,city,state,postal_code,region,product_id,category,sub_category,product_name,sales,quantity,discount,profit
4.0,US-2019-108966,2019-10-11,2019-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,Florida,33311.0,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,5.0,0.45,-383.03100000000006
5.0,US-2019-108966,2019-10-11,2019-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,Florida,33311.0,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.368000000000002,2.0,0.2,2.516399999999999
24.0,US-2021-156909,2021-07-16,2021-07-18,Second Class,SF-20065,Sandra Flanagan,Consumer,United States,None,Pennsylvania,19140.0,East,FUR-CH-10002774,Furniture,Chairs,"Global Deluxe Stacking Chair, Gray",71.37199999999999,2.0,0.3,-1.0196000000000005


In [25]:
%%sql
-- starts with S
SELECT TOP 2 *
FROM orders
WHERE customer_name LIKE 's%'

 * mssql+pyodbc:///?odbc_connect=DRIVER%3D%7BODBC+Driver+17+for+SQL+Server%7D%3BSERVER%3DSANDIPAN%5CSQLEXPRESS%3BDATABASE%3Dmaster%3BTrusted_Connection%3Dyes%3B
Done.


row_id,order_id,order_date,ship_date,ship_mode,customer_id,customer_name,segment,country,city,state,postal_code,region,product_id,category,sub_category,product_name,sales,quantity,discount,profit
4.0,US-2019-108966,2019-10-11,2019-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,Florida,33311.0,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,5.0,0.45,-383.03100000000006
5.0,US-2019-108966,2019-10-11,2019-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,Florida,33311.0,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.368000000000002,2.0,0.2,2.516399999999999


In [27]:
%%sql
-- contains "an" anywhere
SELECT TOP 2 *
FROM orders
WHERE customer_name LIKE '%an%'

 * mssql+pyodbc:///?odbc_connect=DRIVER%3D%7BODBC+Driver+17+for+SQL+Server%7D%3BSERVER%3DSANDIPAN%5CSQLEXPRESS%3BDATABASE%3Dmaster%3BTrusted_Connection%3Dyes%3B
Done.


row_id,order_id,order_date,ship_date,ship_mode,customer_id,customer_name,segment,country,city,state,postal_code,region,product_id,category,sub_category,product_name,sales,quantity,discount,profit
3.0,CA-2020-138688,2020-06-12,2020-06-16,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,California,90036.0,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters by Universal,14.62,2.0,0.0,6.8713999999999995
4.0,US-2019-108966,2019-10-11,2019-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,Florida,33311.0,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,5.0,0.45,-383.03100000000006


In [34]:
%%sql
-- exactly 5 characters
SELECT TOP 2 *
FROM orders
WHERE customer_name LIKE '_____'

-- No Output as there is no name of exact 5 character

 * mssql+pyodbc:///?odbc_connect=DRIVER%3D%7BODBC+Driver+17+for+SQL+Server%7D%3BSERVER%3DSANDIPAN%5CSQLEXPRESS%3BDATABASE%3Dmaster%3BTrusted_Connection%3Dyes%3B
0 rows affected.


row_id,order_id,order_date,ship_date,ship_mode,customer_id,customer_name,segment,country,city,state,postal_code,region,product_id,category,sub_category,product_name,sales,quantity,discount,profit


In [35]:
%%sql
-- starts with S or M
SELECT TOP 2 *
FROM orders
WHERE customer_name LIKE '[SM]%'

 * mssql+pyodbc:///?odbc_connect=DRIVER%3D%7BODBC+Driver+17+for+SQL+Server%7D%3BSERVER%3DSANDIPAN%5CSQLEXPRESS%3BDATABASE%3Dmaster%3BTrusted_Connection%3Dyes%3B
Done.


row_id,order_id,order_date,ship_date,ship_mode,customer_id,customer_name,segment,country,city,state,postal_code,region,product_id,category,sub_category,product_name,sales,quantity,discount,profit
4.0,US-2019-108966,2019-10-11,2019-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,Florida,33311.0,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,5.0,0.45,-383.03100000000006
5.0,US-2019-108966,2019-10-11,2019-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,Florida,33311.0,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.368000000000002,2.0,0.2,2.516399999999999


In [36]:
%%sql
-- does NOT start with S or M

SELECT TOP 2 *
FROM orders
WHERE customer_name NOT LIKE '[SM]%' 

 * mssql+pyodbc:///?odbc_connect=DRIVER%3D%7BODBC+Driver+17+for+SQL+Server%7D%3BSERVER%3DSANDIPAN%5CSQLEXPRESS%3BDATABASE%3Dmaster%3BTrusted_Connection%3Dyes%3B
Done.


row_id,order_id,order_date,ship_date,ship_mode,customer_id,customer_name,segment,country,city,state,postal_code,region,product_id,category,sub_category,product_name,sales,quantity,discount,profit
1.0,CA-2020-152156,2020-11-08,2020-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,42420.0,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.96,2.0,0.0,41.9136
2.0,CA-2020-152156,2020-11-08,2020-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,42420.0,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs, Rounded Back",731.9399999999999,3.0,0.0,219.58199999999997


## 6. ORDER BY

**Definition:**

- Sorts the final result.
- Default direction is ascending (`ASC`) if you don't specify `ASC` or `DESC`.
- Multiple sort columns are applied left to right — ties on the first column are broken by the second, and so on.

In [40]:
%%sql
SELECT TOP 3 * 
FROM orders 
ORDER BY sales DESC;

 * mssql+pyodbc:///?odbc_connect=DRIVER%3D%7BODBC+Driver+17+for+SQL+Server%7D%3BSERVER%3DSANDIPAN%5CSQLEXPRESS%3BDATABASE%3Dmaster%3BTrusted_Connection%3Dyes%3B
Done.


row_id,order_id,order_date,ship_date,ship_mode,customer_id,customer_name,segment,country,city,state,postal_code,region,product_id,category,sub_category,product_name,sales,quantity,discount,profit
2698.0,CA-2018-145317,2018-03-18,2018-03-23,Standard Class,SM-20320,Sean Miller,Home Office,United States,Jacksonville,Florida,32216.0,South,TEC-MA-10002412,Technology,Machines,Cisco TelePresence System EX90 Videoconferencing Unit,22638.48,6.0,0.5,-1811.0784000000021
6827.0,CA-2020-118689,2020-10-02,2020-10-09,Standard Class,TC-20980,Tamara Chand,Corporate,United States,Lafayette,Indiana,47905.0,Central,TEC-CO-10004722,Technology,Copiers,Canon imageCLASS 2200 Advanced Copier,17499.949999999997,5.0,0.0,8399.975999999999
8154.0,CA-2021-140151,2021-03-23,2021-03-25,First Class,RB-19360,Raymond Buch,Consumer,United States,Seattle,Washington,98115.0,West,TEC-CO-10004722,Technology,Copiers,Canon imageCLASS 2200 Advanced Copier,13999.96,4.0,0.0,6719.980799999999


In [45]:
%%sql
-- Sort orders by city, and within each city, show the highest sales first.

SELECT TOP 2 * 
FROM orders 
ORDER BY city ASC, sales DESC;

 * mssql+pyodbc:///?odbc_connect=DRIVER%3D%7BODBC+Driver+17+for+SQL+Server%7D%3BSERVER%3DSANDIPAN%5CSQLEXPRESS%3BDATABASE%3Dmaster%3BTrusted_Connection%3Dyes%3B
Done.


row_id,order_id,order_date,ship_date,ship_mode,customer_id,customer_name,segment,country,city,state,postal_code,region,product_id,category,sub_category,product_name,sales,quantity,discount,profit
14.0,CA-2020-161389,2020-12-05,2020-12-10,Standard Class,IM-15070,Irene Maddox,Consumer,United States,None,Washington,98103.0,West,OFF-BI-10003656,Office Supplies,Binders,Fellowes PB200 Plastic Comb Binding Machine,407.97600000000006,3.0,0.2,132.59219999999993
24.0,US-2021-156909,2021-07-16,2021-07-18,Second Class,SF-20065,Sandra Flanagan,Consumer,United States,None,Pennsylvania,19140.0,East,FUR-CH-10002774,Furniture,Chairs,"Global Deluxe Stacking Chair, Gray",71.37199999999999,2.0,0.3,-1.0196000000000005


## 7. Aggregate Functions

**Definition:**

- `COUNT(*)` — counts every row, NULLs included.
- `COUNT(col)` — counts only rows where `col` is NOT NULL.
- `COUNT(DISTINCT col)` — counts unique non-NULL values.
- `SUM`, `AVG`, `MIN`, `MAX` — all ignore NULL values in the column they're working on.

In [46]:
%%sql
SELECT
    COUNT(*)         AS total_rows,
    COUNT(discount)   AS rows_with_discount,
    SUM(sales)        AS total_sales,
    AVG(sales)        AS avg_sales,
    MIN(profit)       AS min_profit,
    MAX(profit)       AS max_profit
FROM orders;

 * mssql+pyodbc:///?odbc_connect=DRIVER%3D%7BODBC+Driver+17+for+SQL+Server%7D%3BSERVER%3DSANDIPAN%5CSQLEXPRESS%3BDATABASE%3Dmaster%3BTrusted_Connection%3Dyes%3B
Done.


total_rows,rows_with_discount,total_sales,avg_sales,min_profit,max_profit
9994,9994,2297200.8602999584,229.85800083049415,-6599.978000000001,8399.975999999999


In [53]:
%%sql

SELECT SUM(sales) / COUNT(sales) AS avg_sales  
FROM orders;

 * mssql+pyodbc:///?odbc_connect=DRIVER%3D%7BODBC+Driver+17+for+SQL+Server%7D%3BSERVER%3DSANDIPAN%5CSQLEXPRESS%3BDATABASE%3Dmaster%3BTrusted_Connection%3Dyes%3B
Done.


avg_sales
229.85800083049415


In [54]:
%%sql

SELECT ROUND(SUM(sales) / COUNT(sales),2) AS avg_sales  
FROM orders;

 * mssql+pyodbc:///?odbc_connect=DRIVER%3D%7BODBC+Driver+17+for+SQL+Server%7D%3BSERVER%3DSANDIPAN%5CSQLEXPRESS%3BDATABASE%3Dmaster%3BTrusted_Connection%3Dyes%3B
Done.


avg_sales
229.86


## 8. GROUP BY and HAVING

**Definition:**

- `GROUP BY` — puts rows into groups based on column values.
- `HAVING` — filters out whole groups after the grouping/aggregation has already happened.
- `WHERE` filters rows before grouping; `HAVING` filters groups after grouping. This is the only place an aggregate function is allowed as a filter condition.

**Example:**

- "Show categories where total sales are over 10,000" needs `HAVING`, because the filter condition (`SUM(sales) > 10000`) depends on an aggregate.

In [55]:
%%sql

SELECT category, SUM(sales) AS total_sales
FROM orders
GROUP BY category
HAVING SUM(sales) > 10000;

 * mssql+pyodbc:///?odbc_connect=DRIVER%3D%7BODBC+Driver+17+for+SQL+Server%7D%3BSERVER%3DSANDIPAN%5CSQLEXPRESS%3BDATABASE%3Dmaster%3BTrusted_Connection%3Dyes%3B
Done.


category,total_sales
Office Supplies,719047.0320000001
Furniture,741999.7952999983
Technology,836154.0329999967


## 9. DISTINCT vs GROUP BY

**Definition:**

- `DISTINCT` — removes duplicate rows, no aggregation involved.
- `GROUP BY` — forms groups, almost always paired with an aggregate function, and is more powerful when you need per-group calculations rather than just uniqueness.

**Example:** - "List Top 3 city along with its total sales"

In [60]:
%%sql

SELECT TOP 3 city, ROUND(SUM(sales),1) AS Total_Sales
FROM orders
GROUP BY city
ORDER BY Total_Sales DESC

 * mssql+pyodbc:///?odbc_connect=DRIVER%3D%7BODBC+Driver+17+for+SQL+Server%7D%3BSERVER%3DSANDIPAN%5CSQLEXPRESS%3BDATABASE%3Dmaster%3BTrusted_Connection%3Dyes%3B
Done.


city,Total_Sales
New York City,256368.2
Los Angeles,175851.3
Seattle,119132.8
